### Packages

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from IPython.display import display

pd.set_option('display.max_columns', None)
np.random.seed(1)

### Data

In [ ]:
# Load CSV files

# Test Data
testop = pd.read_csv('../data/2024-34-1/data/test_operational_readouts.csv')
testspec = pd.read_csv('../data/2024-34-1/data/test_specifications.csv')
# Train Data
trainop = pd.read_csv('../data/2024-34-1/data/train_operational_readouts.csv')
trainspec = pd.read_csv('../data/2024-34-1/data/train_specifications.csv')
traintte = pd.read_csv('../data/2024-34-1/data/train_tte.csv')
# Validation Data
vallab = pd.read_csv('../data/2024-34-1/data/validation_labels.csv')
valop = pd.read_csv('../data/2024-34-1/data/validation_operational_readouts.csv')
valspec = pd.read_csv('../data/2024-34-1/data/validation_specifications.csv')

# Convert to DataFrames

# Test Data
df_testop = pd.DataFrame(data=testop)
df_testspec = pd.DataFrame(data=testspec)
# Train Data
df_trainop = pd.DataFrame(data=trainop)
df_trainspec = pd.DataFrame(data=trainspec)
df_traintte = pd.DataFrame(data=traintte)
# Validation Data
df_vallab = pd.DataFrame(data=vallab)
df_valop = pd.DataFrame(data=valop)
df_valspec = pd.DataFrame(data=valspec)

In [ ]:
print(df_trainop.shape)
df_trainop.head()

In [ ]:
print(df_trainspec.shape)
df_trainspec.head()

In [ ]:
print(df_traintte.shape)
df_traintte.head()

In [ ]:
def add_temporal_aggregates(df_indexed, sensor_cols):
    """
    Adds expanding-window temporal aggregate features (last, mean, std,
    slope, delta) for every sensor column, computed per vehicle using
    only that vehicle's readings up to and including each row -- never
    future readings, to avoid leakage into earlier-timestep labels.

    Performance note: slope is computed via closed-form cumulative sums
    (a standard least-squares trick) instead of .expanding().apply(),
    which would call a Python function once per row per sensor -- far
    too slow at this dataset's scale (1.1M+ rows x 105 sensors) and
    the kind of thing that previously crashed the kernel on this data.

    Args:
        df_indexed (DataFrame): operational readouts, MUST already be
            sorted by vehicle_id then time_step (expanding logic assumes
            row order == chronological order within each vehicle).
        sensor_cols (list of str): sensor column names to aggregate.

    Returns:
        df_indexed (DataFrame): same dataframe, with 5 new columns added
            per sensor (e.g. "397_0_last", "397_0_mean", "397_0_std",
            "397_0_slope", "397_0_delta").
    """
    grouped_by_vehicle = df_indexed.groupby(level=0)

    for sensor in sensor_cols:
        values = df_indexed[sensor]
        valid = values.notna()

        # last: just the raw reading itself, no computation needed
        df_indexed[f"{sensor}_last"] = values

        #  mean / std: pandas' built-in expanding ops already skip
        # NaNs correctly, and are implemented in fast compiled code
        df_indexed[f"{sensor}_mean"] = grouped_by_vehicle[sensor].expanding().mean().values
        df_indexed[f"{sensor}_std"] = grouped_by_vehicle[sensor].expanding().std().values

        #  delta: current value minus this vehicle's very first
        first_value = df_indexed.groupby(level=0)[sensor].transform("first")
        df_indexed[f"{sensor}_delta"] = values - first_value

        # slope: expanding least-squares slope of sensor value vs.
        time_step = df_indexed["time_step"]

        # zero out invalid (NaN-sensor) rows' contribution to the sums,
        # so missing readings are skipped rather than treated as 0
        x = time_step.where(valid, 0)
        y = values.where(valid, 0)
        xy = (time_step * values).where(valid, 0)
        xx = (time_step ** 2).where(valid, 0)
        n = valid.astype(int)  # counts only valid readings toward n

        # cumulative sums per vehicle, using only valid readings so far
        cum_n = grouped_by_vehicle[n.name].cumsum() if hasattr(n, "name") else n.groupby(df_indexed.index).cumsum()

        tmp = pd.DataFrame({
            "n": n, "x": x, "y": y, "xy": xy, "xx": xx
        }, index=df_indexed.index)
        cum = tmp.groupby(level=0).cumsum()

        # standard least-squares slope formula, applied elementwise:
        # slope = (n*sum(xy) - sum(x)*sum(y)) / (n*sum(xx) - sum(x)^2)
        numerator = cum["n"] * cum["xy"] - cum["x"] * cum["y"]
        denominator = cum["n"] * cum["xx"] - cum["x"] ** 2

        slope = numerator / denominator

        slope[(cum["n"] < 2) | (denominator == 0)] = np.nan

        df_indexed[f"{sensor}_slope"] = slope

        df_indexedtrainop_idx = df_trainop.set_index("vehicle_id").sort_index()

        return df_indexedtrainop_idx


trainop_idx = df_trainop.set_index("vehicle_id").sort_index()
testop_idx = df_testop.set_index("vehicle_id").sort_index()
valop_idx = df_valop.set_index("vehicle_id").sort_index()


def getsetname(df_indexed):
    """ Provide the name of the set type for string interpolation.
    Args:
        df_indexed (): Pandas DataFrame index to the corresponding dataset.
    Returns:
        (String): The name of the dataset as a string.
    """

    if df_indexed is trainop_idx:
        return "training set"
    elif df_indexed is testop_idx:
        return "test set"
    elif df_indexed is valop_idx:
        return "validation set"



def sensor_readouts(df_indexed, graph_type, vehicle_id=None):
    """Produce a multi-variate time series chart that displays sensor readouts over time...
        until the vehicle is outside of its monitoring window.
    Args:
        vehicle_id (Integer): Target for Pandas DataFrame to extract a vehicle's unique sensor data.
    """
    Set = getsetname(df_indexed)

    if vehicle_id is not None and vehicle_id not in df_indexed.index:
        print(
            f"Invalid Vehichle ID: Vehicle ID {vehicle_id} is either out of range or not included in the {set} set."
        )
        return None

    full_data = df_indexed.reset_index()

    if vehicle_id is not None:
        df = df_indexed.loc[[vehicle_id]].reset_index()
        data = df.iloc[:, 1:]

    sensors = [c for c in full_data.iloc[:, 2:].columns]
    cols = [c for c in full_data.iloc[:, 1:].columns]

    for c in cols:
        if full_data[c].isna().all():
            continue
        col_min, col_max = np.nanmin(full_data[c]), np.nanmax(full_data[c])
        if col_max == col_min:
            continue
        if vehicle_id is not None:
            data[c] = (data[c] - col_min) / (col_max - col_min)
        else:
            full_data[c] = (full_data[c] - col_min) / (col_max - col_min)

    if vehicle_id is not None:
        long_df_single = data.melt(
            id_vars="time_step",
            value_vars=sensors,
            var_name="sensor",
            value_name="value",
        )

        fig0 = px.line(
            long_df_single,
            x="time_step",
            y="value",
            color="sensor",
            title=f"{Set.title()} Sensor Readouts | Vehicle ID {vehicle_id}",
            render_mode="webgl",
            template="plotly_dark"
        )
        fig0.update_layout(height=600)

        fig1 = px.imshow(
            data[sensors].corr(),
            x=sensors,
            y=sensors,
            labels={"x": "sensor", "y": "sensor", "color": "correlation"},
            template="plotly_dark",
            title=f"{Set.title()} Sensor Readouts | Vehicle ID {vehicle_id}",
        )
        fig1.update_layout(height=800)

    else:
        long_df_all = full_data.melt(
            id_vars="time_step",
            value_vars=sensors,
            var_name="sensor",
            value_name="value",
        )

        agg = long_df_all.groupby(["time_step", "sensor"], as_index=False)[
            "value"
        ].mean()
        wide = agg.pivot(index="time_step", columns="sensor", values="value")

        fig2 = px.line(
            agg,
            x="time_step",
            y="value",
            color="sensor",
            title=f"{Set.title()} Sensor Readouts | All Vehicles",
            render_mode="webgl",
            template="plotly_dark"
        )
        fig2.update_layout(height=600)

        fig3 = px.imshow(
            wide.corr(),
            x=sensors,
            y=sensors,
            labels={"x": "sensor", "y": "sensor", "color": "correlation"},
            template="plotly_dark",
            title=f"{Set.title()} Sensor Readouts Mean Correlation | All Vehicles",
        )
        fig3.update_layout(height=800)

    if graph_type == "line" and vehicle_id is not None:
        fig0.show()
    elif graph_type == "correlation" and vehicle_id is not None:
        fig1.show()
    if graph_type == "line" and vehicle_id is None:
        fig2.show()
    elif graph_type == "correlation" and vehicle_id is None:
        fig3.show()

In [ ]:
# Edit the vehicle ID and specify the set index to plot its individual sensor data overtime.
sensor_readouts(
    trainop_idx,  # trainop_idx, testop_idx or valop_idx
    'line',
    0
)

In [ ]:
# Edit the vehicle ID and specify the set index to compare sensor-to-sensor correlation per vehicle.
sensor_readouts(
    trainop_idx,  # trainop_idx, testop_idx or valop_idx
    'correlation',
    0  # Number between 0 and x
)

In [ ]:
# Edit the set index to see how mean sensor readouts change over timesteps set wide.

sensor_readouts(
    trainop_idx,  # trainop_idx, testop_idx or valop_idx
    'line',
)

In [ ]:
# Edit the set index to see how mean sensor readouts over time correlate sensor to sensor set wide.

sensor_readouts(
    trainop_idx,  # trainop_idx, testop_idx or valop_idx
    'correlation',  # line or correlation
)

#### Findings:
- no sensor readout pairs display a negative correlation
- sensor readouts display consistent NaN values for many vehicles that do not correlate with other sensors
- sensor readouts trend toward linear growth with minimal volatility for individual vehicles
- mean sensor readouts spanning the entire set trend toward linear growth with volatility that increases as time steps and sensor values grow

In [ ]:
def spec_counts_row(dfset):
    """ Finds the unique spec rows in a dataset and describes the distribution of specifications.
    Args:
        dfset (Pandas: DataFrame):
    Returns:
        labels (list): list of placeholder names for row partitions.
        values (list): list of counts for vehicles of the same specification.
    """
    specs = dfset.iloc[:, 1:]  # Drop the vehicle ID column
    specs.drop_duplicates(inplace=True)
    spec_cols = specs.columns.tolist()
    counts = dfset.groupby(spec_cols).size()
    values = [counts.get(tuple(row), 0) for row in specs.itertuples(index=False)]
    labels = [f"fspec_{i}" for i in range(len(specs))]
    return labels, values

In [ ]:
labels, values = spec_counts_row(df_trainspec)

counts = pd.Series(values, index=labels).sort_values(ascending=False)
print(counts.describe())
print((counts.cumsum() / counts.sum()).head(20))

#### findings:
- Parsing unique specification combinations by rows leads to few unique partitions that can not be used to describe any pattern in the data at scale alone. (3000 > unique combinations)

In [ ]:
def spec_counts_col(dfset):
    """ Finds the unique values within each spec column and describes the
        distribution of specifications, per column.
    Args:
        dfset (Pandas: DataFrame):
    Returns:
        results (dict): maps each spec column name to a (labels, values) tuple,
            where labels are placeholder names for that column's unique values
            and values are the counts of vehicles matching each one.
    """
    specs = dfset.iloc[:, 1:]  # Drop the vehicle ID column
    spec_cols = specs.columns.tolist()

    results = {}
    for col in spec_cols:
        counts = dfset[col].value_counts()
        labels = [f"{col}_{i}" for i in range(len(counts))]
        values = counts.tolist()
        results[col] = (labels, values)

    return results

In [ ]:

for col, (labels, values) in spec_counts_col(df_trainspec).items():
    counts = pd.Series(values, index=labels).sort_values(ascending=False)
    print(f"--- {col} ---")
    print(counts.describe())
    print((counts.cumsum() / counts.sum()).head(20))

##### Findings:
- Parsing by unique columns causes far fewer sparse pairings than rows